# Step 5C.1 — Dataset audit перед F0/F1 (binary detection, flow-level)
Не будує жодної моделі, не фіксує split. Мета — зібрати фактичні цифри для трьох
ЗБАГАЧЕНИХ (Step 3) датасетів: обсяг, розподіл binary target (Normal/Attack), наявність
реального per-flow часового поля, дублікати, розподіл `cve_ctx_*` (особливо
`cve_ctx_service_in_kev` — потенційний shortcut), і зв'язок target з цією ознакою.

Ядро: IDS Research.

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

ENRICHED_DIR = Path("/srv/ids_research/data/enriched")

DATASETS = {
    "GeNIS": dict(file="genis_with_cve_context.parquet", label_col="SubCategoryLabel",
                  benign_prefix="benign"),
    "CIC-IDS2017": dict(file="cic_ids2017_with_cve_context.parquet", label_col="attack_label",
                        benign_value="BENIGN"),
    "UNSW-NB15": dict(file="unsw_nb15_with_cve_context.parquet", label_col="attack_cat",
                      benign_value="Normal"),
}

loaded = {}
for name, cfg in DATASETS.items():
    path = ENRICHED_DIR / cfg["file"]
    df = pd.read_parquet(path)
    loaded[name] = df
    print(f"{name}: {len(df):,} рядків, {len(df.columns)} колонок — {path}")

GeNIS: 1,472,133 рядків, 140 колонок — /srv/ids_research/data/enriched/genis_with_cve_context.parquet
CIC-IDS2017: 2,827,677 рядків, 96 колонок — /srv/ids_research/data/enriched/cic_ids2017_with_cve_context.parquet
UNSW-NB15: 2,540,047 рядків, 62 колонок — /srv/ids_research/data/enriched/unsw_nb15_with_cve_context.parquet


In [2]:
# ---------- 1. Binary target: Normal vs Attack, за ВЛАСНОЮ таксономією кожного датасету ----------
for name, cfg in DATASETS.items():
    df = loaded[name]
    label_col = cfg["label_col"]
    print(f"\n===== {name}: {label_col} =====")
    print(df[label_col].value_counts())

    if "benign_prefix" in cfg:
        is_attack = ~df[label_col].astype(str).str.lower().str.startswith(cfg["benign_prefix"])
    else:
        is_attack = df[label_col] != cfg["benign_value"]

    df["_is_attack"] = is_attack.astype(int)
    n_normal = int((~is_attack).sum())
    n_attack = int(is_attack.sum())
    print(f"\nBinary target: Normal={n_normal:,} ({n_normal/len(df):.2%})  "
          f"Attack={n_attack:,} ({n_attack/len(df):.2%})")


===== GeNIS: SubCategoryLabel =====
SubCategoryLabel
dos-icmp             393216
dos-udp              393216
dos-pushack          392736
dos-slowloris        168116
dos-hulk              67821
benign-user           18048
benign-background     12862
bruteforce-smb        10001
bruteforce-ssh         8677
benign-admin           4096
bruteforce-ftp         3344
Name: count, dtype: int64

Binary target: Normal=35,006 (2.38%)  Attack=1,437,127 (97.62%)

===== CIC-IDS2017: attack_label =====
attack_label
BENIGN                        2271122
DoS Hulk                       230123
PortScan                       158804
DDoS                           128025
DoS GoldenEye                   10293
FTP-Patator                      7935
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1956
Web Attack – Brute Force         1507
Web Attack – XSS                  652
Infiltration                       36
W

In [3]:
# ---------- 2. Пошук реального per-flow часового поля (не snapshot_date з CVE-контексту) ----------
# УВАГА: "dur" (duration) НЕ є timestamp-полем і навмисно ВИКЛЮЧЕНИЙ з temporal candidates —
# це тривалість flow, а не момент часу, і використання dur замість реального часового поля
# для temporal split було б методологічною помилкою. Regex вимагає точної межі слова
# (^|_)...($|_), щоб "dur"/"duration" не зловилися випадково через підрядок.
# Без жорстких меж слова для часових ключових -> ловить і "StartTime"/"EndTime" (camelCase,
# "time" не на межі слова через префікс), і "start_time"/"Timestamp". DURATION_KEYWORDS
# лишається зі строгими межами слова — інакше зловив би щось на кшталт "endurance".
TIME_KEYWORDS = re.compile(r"(timestamp|datetime|date|starttime|endtime|stime|ltime|start_time|end_time)",
                           re.IGNORECASE)
DURATION_KEYWORDS = re.compile(r"(^|_)(dur|duration)($|_)", re.IGNORECASE)

for name, df in loaded.items():
    print(f"\n===== {name}: кандидати на часове поле (timestamp, НЕ duration) =====")
    candidates = [c for c in df.columns if TIME_KEYWORDS.search(c)]
    if not candidates:
        print("  ⚠ Жодного очевидного часового поля не знайдено за назвою колонки.")
    for c in candidates:
        sample = df[c].dropna()
        print(f"  {c:25s} dtype={str(df[c].dtype):12s} n_unique={df[c].nunique():>10,} "
              f"приклад={sample.iloc[0] if len(sample) else 'N/A'}")

    dur_candidates = [c for c in df.columns if DURATION_KEYWORDS.search(c)]
    if dur_candidates:
        print(f"\n  ℹ Знайдено duration-поля (НЕ timestamp, НЕ використовувати для temporal split): "
              f"{dur_candidates}")



===== GeNIS: кандидати на часове поле (timestamp, НЕ duration) =====
  StartTime                 dtype=datetime64[ms] n_unique=    31,362 приклад=2025-02-10 12:11:17

  ℹ Знайдено duration-поля (НЕ timestamp, НЕ використовувати для temporal split): ['Dur']

===== CIC-IDS2017: кандидати на часове поле (timestamp, НЕ duration) =====
  Timestamp                 dtype=datetime64[ns] n_unique=    27,964 приклад=2017-07-03 08:55:58

===== UNSW-NB15: кандидати на часове поле (timestamp, НЕ duration) =====
  Stime                     dtype=int64        n_unique=    85,348 приклад=1421927414
  Ltime                     dtype=int64        n_unique=    85,361 приклад=1421927414

  ℹ Знайдено duration-поля (НЕ timestamp, НЕ використовувати для temporal split): ['dur']


In [4]:
# ---------- 3. Справжні дублікати рядків + repeated 5-tuple (НЕ "дублікати flow") ----------
# УВАГА: однаковий 5-tuple (src/dst ip, src/dst port, proto) може АБСОЛЮТНО НОРМАЛЬНО
# повторюватися в різні моменти часу (різні окремі з'єднання того самого напрямку) —
# тому це НЕ доказ дублікатів flow без часової компоненти. Називаємо це "repeated 5-tuple"
# (діагностика повторюваних flow identities), а не "дублікати". Справжня перевірка
# дублікатів записів — це ЛИШЕ df.duplicated() на повних рядках.
IDENTITY_KEYWORDS = {
    "src_ip": re.compile(r"^(srcip|src_ip|source_ip|src_addr|srcaddr)$", re.IGNORECASE),
    "dst_ip": re.compile(r"^(dstip|dst_ip|destination_ip|dst_addr|dstaddr)$", re.IGNORECASE),
    "src_port": re.compile(r"^(sport|src_port|source_port)$", re.IGNORECASE),
    "dst_port": re.compile(r"^(dsport|dst_port|destination_port)$", re.IGNORECASE),
    "proto": re.compile(r"^(proto|protocol)$", re.IGNORECASE),
}

for name, df in loaded.items():
    print(f"\n===== {name}: повні дублікати рядків =====")
    n_full_dup = df.duplicated().sum()
    print(f"  Повні дублікати рядків (усі колонки, СПРАВЖНІ дублікати): {n_full_dup:,} "
          f"({n_full_dup/len(df):.2%})")

    identity_cols = {}
    for key, pattern in IDENTITY_KEYWORDS.items():
        match = [c for c in df.columns if pattern.match(c)]
        if match:
            identity_cols[key] = match[0]

    print(f"\n===== {name}: repeated 5-tuple (діагностика, НЕ доказ дублікатів без часу) =====")
    if len(identity_cols) >= 3:
        cols = list(identity_cols.values())
        n_repeated = df.duplicated(subset=cols).sum()
        print(f"  Знайдено identity-колонки: {identity_cols}")
        print(f"  Repeated 5-tuple count ({cols}): {n_repeated:,} ({n_repeated/len(df):.2%})")
        print(f"  ℹ Для майбутнього split знадобиться 5-tuple + реальне часове поле (крок 2 вище),")
        print(f"    або dataset-specific flow identifier — саме repeated 5-tuple тут НЕ означає")
        print(f"    duplicate flows.")
    else:
        print(f"  ⚠ Недостатньо identity-колонок для repeated-5-tuple перевірки "
              f"(знайдено лише: {identity_cols})")



===== GeNIS: повні дублікати рядків =====
  Повні дублікати рядків (усі колонки, СПРАВЖНІ дублікати): 0 (0.00%)

===== GeNIS: repeated 5-tuple (діагностика, НЕ доказ дублікатів без часу) =====
  Знайдено identity-колонки: {'src_ip': 'SrcAddr', 'dst_ip': 'DstAddr', 'src_port': 'Sport', 'proto': 'Proto'}
  Repeated 5-tuple count (['SrcAddr', 'DstAddr', 'Sport', 'Proto']): 1,297,831 (88.16%)
  ℹ Для майбутнього split знадобиться 5-tuple + реальне часове поле (крок 2 вище),
    або dataset-specific flow identifier — саме repeated 5-tuple тут НЕ означає
    duplicate flows.

===== CIC-IDS2017: повні дублікати рядків =====
  Повні дублікати рядків (усі колонки, СПРАВЖНІ дублікати): 0 (0.00%)

===== CIC-IDS2017: repeated 5-tuple (діагностика, НЕ доказ дублікатів без часу) =====
  Знайдено identity-колонки: {'src_ip': 'source_ip', 'dst_ip': 'destination_ip', 'src_port': 'source_port', 'dst_port': 'destination_port', 'proto': 'protocol'}
  Repeated 5-tuple count (['source_ip', 'destination_ip'

In [5]:
# ---------- 4. Розподіл cve_ctx_* — особливо cve_ctx_service_in_kev (потенційний shortcut) ----------
CVE_CTX_GROUPS = {
    "CTI/service context": ["cve_ctx_service", "cve_ctx_pool_size", "cve_ctx_context_available"],
    "EPSS context": ["cve_ctx_n_epss", "cve_ctx_epss_coverage", "cve_ctx_epss_available",
                     "cve_ctx_epss_is_retrospective", "cve_ctx_use_epss", "cve_ctx_service_max_epss"],
    "KEV context": ["cve_ctx_service_in_kev"],
    "Product matching": ["cve_ctx_n_generic_os_matches", "cve_ctx_n_specific_product_matches"],
}

for name, df in loaded.items():
    print(f"\n===== {name}: cve_ctx_* огляд =====")
    for group, cols in CVE_CTX_GROUPS.items():
        present = [c for c in cols if c in df.columns]
        missing = [c for c in cols if c not in df.columns]
        print(f"  [{group}] присутні: {present}" + (f"  ВІДСУТНІ: {missing}" if missing else ""))

    if "cve_ctx_service_in_kev" in df.columns:
        print(f"\n  cve_ctx_service_in_kev розподіл:")
        print(df["cve_ctx_service_in_kev"].value_counts(dropna=False))
    if "cve_ctx_use_epss" in df.columns:
        print(f"\n  cve_ctx_use_epss: {df['cve_ctx_use_epss'].mean():.2%} потоків")


===== GeNIS: cve_ctx_* огляд =====
  [CTI/service context] присутні: ['cve_ctx_service', 'cve_ctx_pool_size', 'cve_ctx_context_available']
  [EPSS context] присутні: ['cve_ctx_n_epss', 'cve_ctx_epss_coverage', 'cve_ctx_epss_available', 'cve_ctx_epss_is_retrospective', 'cve_ctx_use_epss', 'cve_ctx_service_max_epss']
  [KEV context] присутні: ['cve_ctx_service_in_kev']
  [Product matching] присутні: ['cve_ctx_n_generic_os_matches', 'cve_ctx_n_specific_product_matches']

  cve_ctx_service_in_kev розподіл:
cve_ctx_service_in_kev
False    821233
True     650900
Name: count, dtype: int64

  cve_ctx_use_epss: 46.16% потоків

===== CIC-IDS2017: cve_ctx_* огляд =====
  [CTI/service context] присутні: ['cve_ctx_service', 'cve_ctx_pool_size', 'cve_ctx_context_available']
  [EPSS context] присутні: ['cve_ctx_n_epss', 'cve_ctx_epss_coverage', 'cve_ctx_epss_available', 'cve_ctx_epss_is_retrospective', 'cve_ctx_use_epss', 'cve_ctx_service_max_epss']
  [KEV context] присутні: ['cve_ctx_service_in_kev

In [6]:
# ---------- 5. Потенційний shortcut: binary target vs cve_ctx_* (KEV/EPSS/availability) ----------
# Той самий принцип, що й dominant_service_share для CVE-рівня (Step 3), але тепер
# безпосередньо для binary Normal/Attack target. Перевіряємо НЕ ЛИШЕ KEV/EPSS-значення,
# а й самі прапорці ДОСТУПНОСТІ контексту: якщо, наприклад, Attack майже завжди має
# context_available=1, а Normal — 0, F1 міг би отримати shortcut через сам факт наявності
# CVE-мапінгу (coverage), а не через реальний ризик-сигнал.
SHORTCUT_CHECK_COLS = ["cve_ctx_context_available", "cve_ctx_epss_available",
                       "cve_ctx_use_epss", "cve_ctx_service_in_kev"]

for name, df in loaded.items():
    print(f"\n===== {name}: target vs cve_ctx_* (shortcut-перевірка) =====")
    for col in SHORTCUT_CHECK_COLS:
        if col in df.columns:
            print(f"\n{col}:")
            print(df.groupby("_is_attack")[col].agg(["count", "mean", "nunique"]))
        else:
            print(f"\n{col}: колонка відсутня")

    print(f"\n--- Overlap cve_ctx_service між Normal і Attack ---")
    if "cve_ctx_service" in df.columns:
        normal_services = set(df.loc[df["_is_attack"] == 0, "cve_ctx_service"].dropna().unique())
        attack_services = set(df.loc[df["_is_attack"] == 1, "cve_ctx_service"].dropna().unique())
        overlap = normal_services & attack_services
        print(f"  Normal services: {len(normal_services):,}")
        print(f"  Attack services: {len(attack_services):,}")
        print(f"  Shared services: {len(overlap):,}")
        if len(normal_services) and len(attack_services):
            jaccard = len(overlap) / len(normal_services | attack_services)
            if jaccard < 0.3:
                interpretation = ("низький overlap сервісів між Normal і Attack означає "
                                  "ПОТЕНЦІЙНИЙ shortcut-ризик для F1 через service identity "
                                  "(не доведений факт, а сигнал для окремої перевірки)")
            elif jaccard > 0.7:
                interpretation = ("високий overlap -> сервіси переважно спільні для обох "
                                  "класів -> нижчий апріорний ризик shortcut через сам факт сервісу")
            else:
                interpretation = "проміжний overlap -> потребує окремого розгляду"
            print(f"  Jaccard overlap: {jaccard:.2%} ({interpretation})")



===== GeNIS: target vs cve_ctx_* (shortcut-перевірка) =====

cve_ctx_context_available:
              count      mean  nunique
_is_attack                            
0             35006  0.825601        2
1           1437127  0.452775        2

cve_ctx_epss_available:
              count      mean  nunique
_is_attack                            
0             35006  0.825601        2
1           1437127  0.452775        2

cve_ctx_use_epss:
              count      mean  nunique
_is_attack                            
0             35006  0.825601        2
1           1437127  0.452775        2

cve_ctx_service_in_kev:
              count      mean  nunique
_is_attack                            
0             35006  0.349254        2
1           1437127  0.444410        2

--- Overlap cve_ctx_service між Normal і Attack ---
  Normal services: 13
  Attack services: 6
  Shared services: 5
  Jaccard overlap: 35.71% (проміжний overlap -> потребує окремого розгляду)

===== CIC-IDS2017: targe

In [7]:
# ---------- ПІДСУМОК Step 5C.1 ----------
print("=" * 70)
print("ПІДСУМОК — що встановлено цим аудитом (без жодного split/моделі)")
print("=" * 70)
for name, df in loaded.items():
    n_normal = int((df["_is_attack"] == 0).sum())
    n_attack = int((df["_is_attack"] == 1).sum())
    print(f"\n{name}: {len(df):,} flows, Normal={n_normal:,} ({n_normal/len(df):.1%}), "
          f"Attack={n_attack:,} ({n_attack/len(df):.1%})")
print("\nНаступний крок (НЕ виконується тут): визначити group/temporal split за реальними")
print("часовими полями (крок 2 вище) і flow-identity дублікатами (крок 3), перш ніж")
print("фіксувати train/val/test і склад F0/F1.")

ПІДСУМОК — що встановлено цим аудитом (без жодного split/моделі)

GeNIS: 1,472,133 flows, Normal=35,006 (2.4%), Attack=1,437,127 (97.6%)

CIC-IDS2017: 2,827,677 flows, Normal=2,271,122 (80.3%), Attack=556,555 (19.7%)

UNSW-NB15: 2,540,047 flows, Normal=2,218,764 (87.4%), Attack=321,283 (12.6%)

Наступний крок (НЕ виконується тут): визначити group/temporal split за реальними
часовими полями (крок 2 вище) і flow-identity дублікатами (крок 3), перш ніж
фіксувати train/val/test і склад F0/F1.
